baseline: a CNN looks at the photo and predicts both labels, the dice count and the pip sum.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image

torch.manual_seed(2026)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


In [2]:
train_csv = "train_labels.csv"
test_csv = "test.csv"
train_dir = "train"
test_dir = "test"

img_size = 192
batch_size = 32
epochs = 30
lr = 3e-4
n_val = 30

In [3]:
labels = pd.read_csv(train_csv)
test_ids = pd.read_csv(test_csv)["image_id"].tolist()

def load(path):
    img = Image.open(path).resize((img_size, img_size))
    return torch.from_numpy(np.array(img)).float().permute(2, 0, 1) / 255

X_train = torch.stack([load(f"{train_dir}/{i}.jpg") for i in labels.image_id])
X_test = torch.stack([load(f"{test_dir}/{i}.jpg") for i in test_ids])
y_count = torch.tensor(labels.n_dice.values).float()
y_sum = torch.tensor(labels.total_sum.values).float()
Y = torch.stack([y_count, y_sum], dim=1)   # two targets per photo

print(X_train.shape, X_test.shape, Y.shape)

torch.Size([200, 3, 192, 192]) torch.Size([50, 3, 192, 192]) torch.Size([200, 2])


In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Linear(256, 2)   # output 0 = count, output 1 = sum

    def forward(self, x):
        x = self.features(x).mean(dim=(2, 3))   # global average pool
        return self.head(x)

model = CNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.L1Loss()

In [5]:
perm = torch.randperm(len(labels))
val_idx, tr_idx = perm[:n_val], perm[n_val:]

model.train()
for epoch in range(1, epochs + 1):
    order = tr_idx[torch.randperm(len(tr_idx))]
    running = 0.0

    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        x, y = X_train[idx].to(device), Y[idx].to(device)
        x = torch.rot90(x, int(torch.randint(4, (1,))), [2, 3])   # tta

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        running += loss.item() * len(idx)

    print(f'epoch {epoch:2d}: train loss {running / len(order):.2f}')

epoch  1: train loss 11.17
epoch  2: train loss 10.35
epoch  3: train loss 9.88
epoch  4: train loss 9.48
epoch  5: train loss 9.12
epoch  6: train loss 8.77
epoch  7: train loss 8.45
epoch  8: train loss 8.13
epoch  9: train loss 7.84
epoch 10: train loss 7.57
epoch 11: train loss 7.30
epoch 12: train loss 7.07
epoch 13: train loss 6.83
epoch 14: train loss 6.58
epoch 15: train loss 6.38
epoch 16: train loss 6.19
epoch 17: train loss 5.98
epoch 18: train loss 5.81
epoch 19: train loss 5.58
epoch 20: train loss 5.39
epoch 21: train loss 5.22
epoch 22: train loss 5.03
epoch 23: train loss 4.85
epoch 24: train loss 4.71
epoch 25: train loss 4.50
epoch 26: train loss 4.35
epoch 27: train loss 4.22
epoch 28: train loss 4.06
epoch 29: train loss 3.87
epoch 30: train loss 3.71


In [6]:
model.eval()
with torch.no_grad():
    val_pred = model(X_train[val_idx].to(device)).cpu()

count_acc = (val_pred[:, 0].round() == y_count[val_idx]).float().mean()
sum_mae = (val_pred[:, 1].round() - y_sum[val_idx]).abs().mean()
mean_mae = (y_sum[tr_idx].mean().round() - y_sum[val_idx]).abs().mean()   # guess the mean

print(f'val sum MAE {sum_mae:.2f}   val count accuracy {count_acc:.0%}   mean-guess MAE {mean_mae:.2f}')

val sum MAE 6.63   val count accuracy 33%   mean-guess MAE 4.77


In [7]:
with torch.no_grad():
    test_pred = model(X_test.to(device)).cpu()

rows = []
for image_id, p in zip(test_ids, test_pred.tolist()):
    rows.append((1, f'count_{image_id}', max(1, round(p[0]))))
    rows.append((2, f'sum_{image_id}', max(1, round(p[1]))))

submission = pd.DataFrame(rows, columns=['subtaskID', 'datapointID', 'answer'])
submission.to_csv('submission.csv', index=False)
print(submission.head())

   subtaskID                datapointID  answer
0          1  count_IMG_20191208_111228      11
1          2    sum_IMG_20191208_111228      24
2          1  count_IMG_20191208_111246      12
3          2    sum_IMG_20191208_111246      27
4          1  count_IMG_20191208_111304      11
